In [1]:
from jax import random
import neural_tangents as nt
from neural_tangents import stax
import jax.numpy as np
import numpy as onp
key = random.PRNGKey(10)
key, x_key, y_key = random.split(key, 3)

In [85]:
test_point = np.array(onp.random.uniform(size=(2, 2, 2, 1)))

In [44]:
test_point

Array([[[[0.49402976],
         [0.4036332 ]],

        [[0.11103132],
         [0.10927278]]]], dtype=float32)

In [68]:
c_init_fn, c_apply_fn, c_kernel_fn = stax.serial(
    stax.Conv(1, (1, 1)),
    stax.Flatten(),
)
key, net_key = random.split(key)
_, c_params = c_init_fn(net_key, (1, 2, 2, 1))

c_ntk_fn = nt.empirical_ntk_fn(f=c_apply_fn, trace_axes=(-1,))

In [74]:
d_init_fn, d_apply_fn, d_kernel_fn = stax.serial(
    stax.Flatten(),
    stax.Dense(4),
)
key, net_key = random.split(key)
_, d_params = d_init_fn(net_key, (1, 2, 2, 1))

d_ntk_fn = nt.empirical_ntk_fn(f=d_apply_fn, trace_axes=(-1,))

In [75]:
new = list(d_params[1])

In [76]:
new[0] = np.eye(4) * c_params[0][0][0][0][0][0]

In [77]:
d_params[1] = tuple(new)

In [78]:
d_params

[(),
 (Array([[0.59212, 0.     , 0.     , 0.     ],
         [0.     , 0.59212, 0.     , 0.     ],
         [0.     , 0.     , 0.59212, 0.     ],
         [0.     , 0.     , 0.     , 0.59212]], dtype=float32),
  None)]

In [79]:
c_params

[(Array([[[[0.59212]]]], dtype=float32), None), ()]

In [86]:
d_ntk_fn(test_point, test_point, d_params)

Array([[0.5559651 , 0.2768467 ],
       [0.2768467 , 0.24656373]], dtype=float32)

In [87]:
c_ntk_fn(test_point, test_point, c_params)

Array([[0.5559652 , 0.2768467 ],
       [0.2768467 , 0.24656373]], dtype=float32)